## 1. Config

In [1]:
# =========================
# CONFIG
# =========================

CONFIG = {
    "size": 100000,

    "word_max_features": 15000,
    "char_max_features": 15000,

    "min_df": 2,

    "word_ngram_range": (1, 2),
    "char_ngram_range": (2, 6),

    "lr": 0.3,
    "epochs": 40,
}

print("CONFIGURATION:")
for k, v in CONFIG.items():
    print(f"{k}: {v}")

CONFIGURATION:
size: 100000
word_max_features: 15000
char_max_features: 15000
min_df: 2
word_ngram_range: (1, 2)
char_ngram_range: (2, 6)
lr: 0.3
epochs: 40


## 2. Imports

In [2]:
import numpy as np
import pandas as pd

from src.dataloader.dataloader import SentenceDataModule
from src.text_embedding.tfidf import TfIdfVectorizerNumpy
from src.models.logreg import SoftmaxLogReg
from src.models.save_model import save_model

## 3. Load dataset

In [3]:
from pathlib import Path

BASE_DIR = Path().resolve().parent  # move out of notebooks/
DATA_PATH = BASE_DIR / "datasets" / "records_long.json"

print("\nLoading dataset...")

dm = SentenceDataModule(
    record_path=str(DATA_PATH),
    size=CONFIG["size"],
    split=(70, 20, 10),
)

train_samples = dm.get_train_loader().samples
val_samples = dm.get_val_loader().samples
test_samples = dm.get_test_loader().samples

print("Train size:", len(train_samples))
print("Val size:", len(val_samples))
print("Test size:", len(test_samples))


Loading dataset...
Sample in dataset 100000
Train size: 70000
Val size: 20000
Test size: 10000


## 4. Extract text + labels

In [4]:
X_train_text = [s["text"] for s in train_samples]
y_train = np.array([s["model"] for s in train_samples])

label_to_id = {
    "human": 0,
    "chatgpt": 1,
    "gemma": 2,
    "anthropic": 3,
    "llama": 4
}

id_to_label = {v: k for k, v in label_to_id.items()}

total = len(y_train)

unique, counts = np.unique(y_train, return_counts=True)

print("\nTraining distribution:")
for u, c in zip(unique, counts):
    pct = (c / total) * 100
    print(f"{id_to_label[u]:10s}: {c:6d} ({pct:5.2f}%)")

X_val_text = [s["text"] for s in val_samples]
y_val = np.array([s["model"] for s in val_samples])

X_test_text = [s["text"] for s in test_samples]
y_test = np.array([s["model"] for s in test_samples])

print("\nExample sample:")
print(X_train_text[0])
print("Label:", y_train[0])


Training distribution:
human     :  14003 (20.00%)
chatgpt   :  13967 (19.95%)
gemma     :  13974 (19.96%)
anthropic :  14022 (20.03%)
llama     :  14034 (20.05%)

Example sample:
Teaching stylistic elements like tone, figurative language and rhetorical devices within varied texts from multiple disciplines gives practice in critical analysis useful for research and assessment. To incorporate diverse writing, instructors can start by auditing existing lessons and standards to identify gaps, supplement core materials with examples contrasting styles across topics, allow choices in writing assignments and dedicate class time for critiquing exemplar texts modeling varied styles. Exposure to diverse styles from an early age can transform how students perceive themselves as thinkers and communicators, ultimately equipping them with versatility and confidence and a lifelong love of writing and self-expression.
Label: 3


## 5.1 WORD TF-IDF

In [5]:
print("\nBuilding WORD TF-IDF...")

word_vectorizer = TfIdfVectorizerNumpy(
    max_features=CONFIG["word_max_features"],
    min_df=CONFIG["min_df"],
    ngram_range=CONFIG["word_ngram_range"],
    analyzer="word",
)

X_train_word = word_vectorizer.fit_transform(X_train_text)
X_val_word = word_vectorizer.transform(X_val_text)
X_test_word = word_vectorizer.transform(X_test_text)

print("Word TF-IDF shape:", X_train_word.shape)


Building WORD TF-IDF...
Word TF-IDF shape: (70000, 15000)


## 5.2 Char TF-IDF

In [6]:
print("\nBuilding CHAR TF-IDF...")

char_vectorizer = TfIdfVectorizerNumpy(
    max_features=CONFIG["char_max_features"],
    min_df=CONFIG["min_df"],
    ngram_range=CONFIG["char_ngram_range"],
    analyzer="char",
)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)
X_test_char = char_vectorizer.transform(X_test_text)

print("Char TF-IDF shape:", X_train_char.shape)


Building CHAR TF-IDF...
Char TF-IDF shape: (70000, 15000)


## 7. Combine features and add length

In [7]:
print("\nCombining features...")

X_train = np.hstack([X_train_word, X_train_char])
X_val = np.hstack([X_val_word, X_val_char])
X_test = np.hstack([X_test_word, X_test_char])

print("Combined shape:", X_train.shape)

print("\nAdding length feature...")

length_train = np.array([len(t) for t in X_train_text]).reshape(-1, 1) / 200
length_val = np.array([len(t) for t in X_val_text]).reshape(-1, 1) / 200
length_test = np.array([len(t) for t in X_test_text]).reshape(-1, 1) / 200

X_train = np.hstack([X_train, length_train])
X_val = np.hstack([X_val, length_val])
X_test = np.hstack([X_test, length_test])

print("Final feature shape:", X_train.shape)


Combining features...
Combined shape: (70000, 30000)

Adding length feature...
Final feature shape: (70000, 30001)


## 8. Train model and save it

In [8]:
print("\nTraining model...")

clf = SoftmaxLogReg(
    input_dim=X_train.shape[1],
    num_classes=5,
    lr=CONFIG["lr"],
)

clf.fit(
    X_train,
    y_train,
    X_val=X_val,
    y_val=y_val,
    epochs=CONFIG["epochs"],
)

MODEL_PATH = Path("../../models/subm2-g5-MEI-A.pkl")

save_model(
    model=clf,
    word_vectorizer=word_vectorizer,
    char_vectorizer=char_vectorizer,
    path=MODEL_PATH,
    config=CONFIG,
    label_to_id=label_to_id,
    save_full_model=True
)


Training model...
Epoch 001 | train_loss=1.2155 train_acc=0.6494 | val_loss=1.2169 val_acc=0.6485
Epoch 002 | train_loss=1.0115 train_acc=0.8270 | val_loss=1.0133 val_acc=0.8246
Epoch 003 | train_loss=0.9106 train_acc=0.7421 | val_loss=0.9137 val_acc=0.7392
Epoch 004 | train_loss=0.8528 train_acc=0.6961 | val_loss=0.8563 val_acc=0.6956
Epoch 005 | train_loss=0.7451 train_acc=0.8363 | val_loss=0.7494 val_acc=0.8304
Epoch 006 | train_loss=0.7020 train_acc=0.8621 | val_loss=0.7064 val_acc=0.8551
Epoch 007 | train_loss=0.6770 train_acc=0.8155 | val_loss=0.6817 val_acc=0.8133
Epoch 008 | train_loss=0.6351 train_acc=0.8840 | val_loss=0.6401 val_acc=0.8789
Epoch 009 | train_loss=0.6175 train_acc=0.8575 | val_loss=0.6211 val_acc=0.8566
Epoch 010 | train_loss=0.5821 train_acc=0.9183 | val_loss=0.5865 val_acc=0.9176
Epoch 011 | train_loss=0.5871 train_acc=0.8698 | val_loss=0.5916 val_acc=0.8672
Epoch 012 | train_loss=0.5670 train_acc=0.8956 | val_loss=0.5723 val_acc=0.8897
Epoch 013 | train_los

## 9. Evaluate

In [9]:
print("\nEvaluating on test set...")

preds = clf.predict(X_test)
acc = np.mean(preds == y_test)

print("Test accuracy:", acc)


Evaluating on test set...
Test accuracy: 0.9509


## 10. Test the given subm1.csv

In [10]:
print("\nLoading CSV...")

df = pd.read_csv(
    str(BASE_DIR) + "/tests/TestData/subm1.csv",
    sep=";",
)

texts = df["Text"].tolist()

print("Number of samples:", len(texts))




print("\nBuilding features for CSV...")

X_word = word_vectorizer.transform(texts)
X_char = char_vectorizer.transform(texts)

X = np.hstack([X_word, X_char])

length = np.array([len(t) for t in texts]).reshape(-1, 1) / 200
X = np.hstack([X, length])

print("CSV feature shape:", X.shape)

label_to_id = {
    "human": 0,
    "chatgpt": 1,
    "gemma": 2,
    "anthropic": 3,
    "llama": 4
}

id_to_company = {
    0: "Human",
    1: "OpenAI",
    2: "Google",
    3: "Anthropic",
    4: "Meta"
}




print("\nRunning predictions...")

preds = clf.predict(X)

df["Label"] = [id_to_company[p] for p in preds]

print(df.head())




df.to_csv("subm2-g5-MEI-A.csv", index=False, sep=";")
print("subm2-g5-MEI-A.csv")


Loading CSV...
Number of samples: 150

Building features for CSV...
CSV feature shape: (150, 30001)

Running predictions...
     ID                                               Text      Label
0  D2-1  A covalent bond is a chemical bond that involv...     OpenAI
1  D2-2  A covalent bond forms when two atoms share one...  Anthropic
2  D2-3  A covalent bond is a type of chemical bond whe...     OpenAI
3  D2-4  A covalent bond is a chemical bond that involv...     OpenAI
4  D2-5  Driven by exciting developments in the field o...  Anthropic
subm2-g5-MEI-A.csv


## 11. Test against subm1_labels_revealed.csv

In [13]:
print("\nLoading CSV...")

df_revealed = pd.read_csv(
    "../subm1_labels_revealed.csv",
    sep=";",
)

df_new_data = pd.read_csv(
    "../subm2-g5-MEI-A.csv",
    sep=";",
)

df_merged = df_revealed.merge(df_new_data, on="ID", suffixes=("_true", "_pred"))

df_new_data.rename(columns={"Label": "Label_pred"}, inplace=True)
df_revealed.rename(columns={"Label": "Label_true"}, inplace=True)

df_merged["correct"] = df_merged["Label_true"] == df_merged["Label_pred"]


accuracy = df_merged["correct"].mean()

print("Accuracy:", accuracy)
print("Correct predictions:", df_merged["correct"].sum())
print("Total samples:", len(df_merged))

df_merged.groupby("Label_true")["correct"].mean()

df_merged.to_csv("subm2-diff-g5-MEI-A.csv", index=False, sep=";")





Loading CSV...
Accuracy: 0.26
Correct predictions: 26
Total samples: 100


In [12]:
print("\nLoading CSV...")

df_revealed = pd.read_csv(
    "../subm1_labels_revealed.csv",
    sep=";",
)

df_old_data = pd.read_csv(
    "../subm1-g5-MEI-A.csv",
    sep=",",
)

df_merged = df_revealed.merge(df_old_data, on="ID", suffixes=("_true", "_pred"))

df_old_data.rename(columns={"Label": "Label_pred"}, inplace=True)
df_revealed.rename(columns={"Label": "Label_true"}, inplace=True)

df_merged["correct"] = df_merged["Label_true"] == df_merged["Label_pred"]


accuracy = df_merged["correct"].mean()

print("Accuracy:", accuracy)
print("Correct predictions:", df_merged["correct"].sum())
print("Total samples:", len(df_merged))

df_merged.groupby("Label_true")["correct"].mean()


Loading CSV...
Accuracy: 0.35
Correct predictions: 35
Total samples: 100


Label_true
Anthropic    0.705882
Google       0.000000
Human        0.617647
Meta         0.055556
OpenAI       0.071429
Name: correct, dtype: float64